In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy.stats import boxcox
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from tqdm import tqdm

In [ ]:
# Constants

KAGGLE_INPUT_BASE_PATH = "/kaggle/input/m5-forecasting-accuracy"
LOCAL_INPUT_BASE_PATH = "./data"
INPUT_BASE_PATH = LOCAL_INPUT_BASE_PATH

ETS_RESULTS_PATH = Path("./results/ets/")
ETS_RESULTS_PATH.mkdir(parents=True, exist_ok=True)

FORECAST_HORIZON = 28
SUBMISSION_F_COLS = [f"F{i}" for i in range(1, FORECAST_HORIZON + 1)]

In [ ]:
# Load data

CALENDAR_DATA = pl.read_csv(f"{INPUT_BASE_PATH}/calendar.csv", try_parse_dates=True)
SALES_TRAIN_EVALUATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_evaluation.csv")
SALES_TRAIN_VALIDATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_validation.csv")
SAMPLE_SUBMISSION = pl.read_csv(f"{INPUT_BASE_PATH}/sample_submission.csv")

In [ ]:
class BoxCoxScaler:
    def __init__(self):
        self._lambda: float | None = None

    @property
    def is_fit(self):
        return self._lambda is not None
    
    def fit_transform(self, y: pl.Series) -> pl.Series:
        y_t, _lambda = boxcox(y.to_numpy())
        self._lambda = _lambda
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)
    
    def transform(self, y: pl.Series) -> pl.Series:
        assert self.is_fit
        y_t = boxcox(y.to_numpy(), lmbda=self._lambda)
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)
    
    def inverse_transform(self, y: pl.Series) -> pl.Series:
        assert self.is_fit

        if self._lambda == 0:
            y_t = np.exp(y.to_numpy())
        else:
            y_t = (y.to_numpy() * self._lambda + 1) ** (1 / self._lambda)
        return pl.Series(name=y.name, values=y_t, dtype=pl.Float32)

### Preprocess

In [ ]:
train_df = (
    SALES_TRAIN_EVALUATION.unpivot(
        index=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
        variable_name="d",
        value_name="sales",
    )
    .join(
        CALENDAR_DATA.select(pl.col("date"), pl.col("d")),
        on="d",
        how="left",
    )
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.UInt64))
)

train_df.head()

### Statsmodels ETS

In [ ]:
FORECASTS_PATH = ETS_RESULTS_PATH / "forecasts"
FORECASTS_PATH.mkdir(parents=True, exist_ok=True)

item_ids: list[str] = train_df["id"].unique().sort().to_list()
for item_id in tqdm(item_ids):
    id_train_df = (
        train_df
        .filter(pl.col("id") == item_id)
        .select(pl.col("id"), pl.col("d"), pl.col("d_index"), pl.col("sales"))
        .sort(by="d_index")
    )
    
    # Define model and fit
    ets_model = ETSModel(
        endog=id_train_df["sales"].to_numpy(),
        error="add",
        seasonal="add",
        seasonal_periods=7,
    )
    fitted_ets_model = ets_model.fit()

    # Get forecasts and save to file.
    item_forecast_df = pl.DataFrame(
        {
            "id": item_id,
            "F_index": pl.arange(1, FORECAST_HORIZON + 1, eager=True),
            "F": [f"F{i}" for i in range(1, FORECAST_HORIZON + 1)],
            "sales": fitted_ets_model.forecast(FORECAST_HORIZON)
            
        }
    )
    item_forecast_df.write_parquet(f"{FORECASTS_PATH / item_id}.pq")

In [ ]:
# Load all forecasts into single df

all_item_forecast_dfs = []
for item_id in item_ids:
    item_forecast_df = pl.read_parquet(FORECASTS_PATH / f"{item_id}.pq")
    all_item_forecast_dfs.append(item_forecast_df)

ets_forecasts_df = pl.concat(all_item_forecast_dfs)

# Prepare for submission
ets_forecasts_df_wide = (
    ets_forecasts_df
    .with_columns(pl.col("sales").round().cast(pl.Int64))
    .pivot(on="F", index="id", values="sales")
)

sample_submission = (
    SAMPLE_SUBMISSION
    .join(ets_forecasts_df_wide, on="id", how="left", suffix="_ets")
    .with_columns(
        pl.coalesce(pl.col(f"{c}_ets"), pl.col(c)).cast(pl.Int64).alias(c)
        for c in SUBMISSION_F_COLS
    )
    .select(SAMPLE_SUBMISSION.columns)
)

# Save to csv
submission_file = FORECASTS_PATH / f"sample_submission_{datetime.now().replace(microsecond=0)}.csv"
sample_submission.write_csv(str(submission_file))

### Nixtla AutoETS